In [2]:
import os
import sys
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Union
from pathlib import Path
import glob
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Add current working directory to Python path
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

print(f"Added to path: {current_dir}")

Added to path: c:\Users\Ondřej Černý\OneDrive - České vysoké učení technické v Praze\Dokumenty\skola\diplomka\dynamic_trend_model


In [3]:
import math
import numpy as np
import matplotlib.pyplot as plt

def plot_histograms_with_stats(
    dfs,
    metric,
    titles=None,
    bins="auto",
    density=False,
    x_range=None,
    ncols=2,
    figsize_per_row=(12, 5),
):
    """
    Plot histograms with stats for a list of DataFrames.

    Parameters
    ----------
    dfs : list of pd.DataFrame
        List of DataFrames to plot.
    metric : str
        Column name to plot.
    titles : list of str, optional
        Titles for each histogram. If None, default titles are used.
    bins : int, sequence, or str
        Histogram bins passed to matplotlib.
    density : bool
        Whether to normalize histogram.
    x_range : tuple, optional
        Shared x-axis range for all histograms. If None, computed from all dfs.
    ncols : int
        Number of histograms per row. Default is 2.
    figsize_per_row : tuple
        Base figure size per row, default (12, 5).
    """

    if not dfs:
        print("No DataFrames provided.")
        return

    for i, df in enumerate(dfs):
        if metric not in df.columns:
            print(f"Incorrect metric name in df index {i}: '{metric}' not found.")
            return

    if titles is None:
        titles = [f"DF {i+1}" for i in range(len(dfs))]
    elif len(titles) != len(dfs):
        print("Length of titles must match length of dfs.")
        return

    values_list = [df[metric].dropna() for df in dfs]

    if all(len(values) == 0 for values in values_list):
        print("All DataFrames have only NaN values for this metric.")
        return

    if x_range is None:
        all_values = np.concatenate([values.to_numpy() for values in values_list if len(values) > 0])
        x_range = (np.min(all_values), np.max(all_values))

    n_plots = len(dfs)
    nrows = math.ceil(n_plots / ncols)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(figsize_per_row[0], figsize_per_row[1] * nrows),
        squeeze=False
    )

    fig.suptitle(metric, fontsize=14)

    def _add_hist_with_stats(ax, values, title):
        ax.hist(
            values,
            bins=bins,
            density=density,
            range=x_range,
            color="skyblue",
            edgecolor="black"
        )
        ax.set_title(title)
        ax.set_xlabel("Value")
        ax.set_ylabel("Density" if density else "Frequency")

        min_v = np.min(values)
        max_v = np.max(values)
        median_v = np.median(values)
        mean_v = np.mean(values)
        std_v = np.std(values)

        stats_lines = [
            f"Min: {min_v:.2f}",
            f"Max: {max_v:.2f}",
            f"Median: {median_v:.2f}",
            f"Mean: {mean_v:.2f}",
            f"Std: {std_v:.2f}",
        ]
        stats_text = "\n".join(stats_lines)

        font_size = max(8, min(10, 10 - len(stats_lines) // 2))

        ax.text(
            0.02,
            0.98,
            stats_text,
            transform=ax.transAxes,
            va="top",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8),
            fontsize=font_size,
        )

    for idx, (values, title) in enumerate(zip(values_list, titles)):
        row = idx // ncols
        col = idx % ncols
        ax = axes[row][col]

        if len(values) == 0:
            ax.set_title(title)
            ax.text(0.5, 0.5, "No data", ha="center", va="center")
            ax.set_axis_off()
        else:
            _add_hist_with_stats(ax, values, title)

    for idx in range(n_plots, nrows * ncols):
        row = idx // ncols
        col = idx % ncols
        axes[row][col].set_axis_off()

    plt.tight_layout()
    plt.show()

In [4]:
def _prepare_time_filtered_df(df, start_date=None, end_date=None):
    """Helper: ensure datetime and filter by dates."""
    df = df.copy()

    if not pd.api.types.is_datetime64_any_dtype(df["timestamp_utc"]):
        df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"])

    if start_date is not None or end_date is not None:
        if isinstance(start_date, str):
            start_date = pd.to_datetime(start_date)
        if isinstance(end_date, str):
            end_date = pd.to_datetime(end_date)

        if start_date is not None:
            df = df[df["timestamp_utc"] >= start_date]
        if end_date is not None:
            df = df[df["timestamp_utc"] <= end_date]

    return df


def plot_water_meter_anomalies(
    df_actuals,
    df_predicted,
    df_red_dots,
    figsize=(14, 6),
    title="Water Meter: Actual vs Predicted Values",
    start_date=None,
    end_date=None,
):
    """
    df_actuals:  columns timestamp_utc, Diff
    df_predicted: columns timestamp_utc, predicted, residual, is_anomaly_predicted, is_anomaly_actual
    df_red_dots: columns timestamp_utc (candidate extra timestamps)
    """

    # --- 3-sigma band based on residuals ---
    if not df_predicted.empty and "residual" in df_predicted.columns:
        res_mean = df_predicted["residual"].mean()
        res_std = df_predicted["residual"].std(ddof=1)
        band = res_mean + 3 * res_std

        df_predicted = df_predicted.copy()
        df_predicted["upper_band"] = df_predicted["predicted"] + band
        df_predicted["lower_band"] = df_predicted["predicted"] - band
    # ---------------------------------------
    
    # Prepare and filter separately
    df_a = _prepare_time_filtered_df(df_actuals, start_date, end_date)
    df_p = _prepare_time_filtered_df(df_predicted, start_date, end_date)
    df_r = _prepare_time_filtered_df(df_red_dots, start_date, end_date)
    df_r_line = pd.DataFrame()

    # Keep only timestamps where Diff is NaN in df_a but not NaN in df_r
    if not df_a.empty and not df_r.empty:
        # Select only needed columns to avoid clashes
        tmp_a = df_a[["timestamp_utc", "Diff"]].rename(columns={"Diff": "Diff_a"})
        tmp_r = df_r[["timestamp_utc", "Diff"]].rename(columns={"Diff": "Diff_r"})
    
        merged = tmp_r.merge(tmp_a, on="timestamp_utc", how="left")
    
        # Condition: Diff_a is NaN (missing in actuals) AND Diff_r is not NaN
        mask = merged["Diff_a"].isna() & merged["Diff_r"].notna()
    
        # Keep only those timestamps in df_r
        df_r = df_r[df_r["timestamp_utc"].isin(merged.loc[mask, "timestamp_utc"])]
        
        df_r_line = df_r[df_r["timestamp_utc"] < df_a["timestamp_utc"].iloc[0]].copy()
        df_r = df_r[df_r["timestamp_utc"] >= df_a["timestamp_utc"].iloc[0]]


    # Update title if zoomed
    if start_date is not None and end_date is not None:
        if isinstance(start_date, str):
            start_date = pd.to_datetime(start_date)
        if isinstance(end_date, str):
            end_date = pd.to_datetime(end_date)
        date_range = (
            f" ({start_date.strftime('%Y-%m-%d %H:%M')} "
            f"to {end_date.strftime('%Y-%m-%d %H:%M')})"
        )
        title = title + date_range

    fig, ax = plt.subplots(figsize=figsize, dpi=100)

    # Actuals (Diff)
    ax.plot(
        df_a["timestamp_utc"],
        df_a["Diff"],
        color="#2E86AB",
        linewidth=2.5,
        marker="o",
        markersize=5,
        label="Actual (Diff)",
        zorder=3,
        alpha=0.9,
    )

    # Predicted
    ax.plot(
        df_p["timestamp_utc"],
        df_p["predicted"],
        color="#A23B72",
        linewidth=2,
        marker="s",
        markersize=4,
        label="Predicted",
        zorder=2,
        alpha=0.8,
        linestyle="--",
    )
    
    # 3-sigma upper/lower band
    if "upper_band" in df_p.columns and "lower_band" in df_p.columns:
        ax.plot(
            df_p["timestamp_utc"],
            df_p["upper_band"],
            color="gray",
            linestyle=":",
            linewidth=1.5,
            label="3σ upper band",
            zorder=1,
            alpha=0.8,
        )
        ax.plot(
            df_p["timestamp_utc"],
            df_p["lower_band"],
            color="gray",
            linestyle=":",
            linewidth=1.5,
            label="3σ lower band",
            zorder=1,
            alpha=0.8,
        )
        # Optional: shaded band
        #ax.fill_between(
        #    df_p["timestamp_utc"],
        #    df_p["lower_band"],
        #    df_p["upper_band"],
        #    color="gray",
        #    alpha=0.15,
        #    zorder=0,
        #)

    # Red dots only for timestamps not in df_actuals
    if not df_r.empty:
        ax.scatter(
            df_r["timestamp_utc"],
            df_r["Diff"],  # or np.nan if you want them detached from y-scale
            color="red",
            s=40,
            marker="o",
            label="Extra timestamps",
            zorder=5,
        )
    
    if not df_r_line.empty:    
        ax.plot(
            df_r_line["timestamp_utc"],
            df_r_line["Diff"],
            color="red",
            linewidth=2.5,
            marker="o",
            markersize=5,
            label="Training Data",
            zorder=3,
            alpha=0.8,
        )

    # Optional anomalies on actuals
    if "is_anomaly_predicted" in df_p.columns:
        anomalies = df_p[df_p["is_anomaly_predicted"] == 1]
        if len(anomalies) > 0:
            ax.scatter(
                anomalies["timestamp_utc"],
                anomalies["actual"],
                color="#F18F01",
                s=150,
                marker="X",
                label="Anomaly (Predicted)",
                zorder=4,
                edgecolors="#C41E3A",
                linewidth=2,
            )
            
    # Optional anomalies on actuals
    if "is_anomaly_actual" in df_p.columns:
        anomalies = df_p[df_p["is_anomaly_actual"] != 0]
        if len(anomalies) > 0:
            ax.scatter(
                anomalies["timestamp_utc"],
                anomalies["actual"],
                color="#FF03CD",
                s=150,
                marker="P",
                label="Anomaly (Actual)",
                zorder=4,
                edgecolors="#C41E3A",
                linewidth=2,
            )

    ax.set_xlabel("Timestamp (UTC)", fontsize=11, fontweight="bold")
    ax.set_ylabel("Water Consumption (Diff)", fontsize=11, fontweight="bold")
    ax.set_title(title, fontsize=13, fontweight="bold", pad=20)

    # Determine global time range from all non-empty dfs
    timestamps = []
    for d in (df_a, df_p, df_r):
        if not d.empty:
            timestamps.append(d["timestamp_utc"].min())
            timestamps.append(d["timestamp_utc"].max())

    if timestamps:
        t_min, t_max = min(timestamps), max(timestamps)
        time_range = (t_max - t_min).total_seconds()
    else:
        time_range = 0

    if time_range < 3600:
        date_format = "%H:%M:%S"
        date_locator = mdates.MinuteLocator(interval=10)
    elif time_range < 86400:
        date_format = "%H:%M"
        date_locator = mdates.HourLocator(interval=2)
    elif time_range < 604800:
        date_format = "%Y-%m-%d %H:%M"
        date_locator = mdates.AutoDateLocator()
    else:
        date_format = "%Y-%m-%d"
        date_locator = mdates.AutoDateLocator()

    ax.xaxis.set_major_formatter(mdates.DateFormatter(date_format))
    ax.xaxis.set_major_locator(date_locator)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

    ax.grid(True, alpha=0.3, linestyle="--", linewidth=0.7)
    ax.set_axisbelow(True)
    ax.legend(loc="best", fontsize=10, framealpha=0.95)
    ax.set_facecolor("#F8F9FA")
    fig.patch.set_facecolor("white")
    fig.tight_layout()

    return fig, ax


In [5]:
"""
Metrics Comparison Toolkit for Water Meter Anomaly Detection Models

This module provides utilities to load, analyze, and compare metrics
across multiple models in Jupyter notebooks with nice pandas table display.

Key Features:
- Load model results from CSV files
- Compute mean and standard deviation of metrics
- Add mean ± std for train / prediction time columns
- Display as formatted pandas DataFrame in Jupyter
- Filter by metric type (all vs non-zero only)

Purpose: Compare multiple anomaly detection models
"""

from pathlib import Path
from typing import List, Union
import glob

import numpy as np
import pandas as pd


# Extra columns for timing information
EXTRA_COLS = [
    "train_train_time_seconds",
    "second_train_time_seconds",
    "prediction_time_seconds",
    "number_of_anomalies",
    "number_of_anomalies_robust"
]


def load_model_results(csv_filepath: str, verbose: bool = True) -> pd.DataFrame:
    """
    Load and filter model results from CSV file.

    Parameters
    ----------
    csv_filepath : str
        Path to CSV file with model results
    verbose : bool
        Print loading status

    Returns
    -------
    pd.DataFrame
        Dataframe with only successful runs
    """
    try:
        df = pd.read_csv(csv_filepath)
        successful = df[df['status'] == 'success'].copy()

        if verbose and len(successful) > 0:
            print(f"✓ Loaded {len(successful)} successful runs")

        return successful
    except Exception as e:
        print(f"✗ Error loading {csv_filepath}: {e}")
        return pd.DataFrame()


def extract_metrics(df: pd.DataFrame, include_non_zero_only: bool = False, unresampled: bool = False) -> pd.DataFrame:
    """
    Extract metric columns from results dataframe.

    Parameters
    ----------
    df : pd.DataFrame
        Results dataframe
    include_non_zero_only : bool
        If True, extract only _nz metrics

    Returns
    -------
    pd.DataFrame
        Dataframe with only metric columns
    """
   
    
    if not unresampled:
        metric_cols = [col for col in df.columns if col.startswith('metric_')]
    else:
        metric_cols = [col for col in df.columns if col.startswith('unresampled_metric_')]

    if include_non_zero_only:
        # Include only non-zero and summary metrics
        metric_cols = [
            col for col in metric_cols
            if col.endswith('_nz') or
               col in [
                   'metric_total_count',
                   'metric_non_zero_count',
                   'metric_zero_count',
                   'metric_non_zero_percentage',
               ]
        ]

    return df[metric_cols]

def compute_metrics_summary(csv_filepaths: List[str],
                            model_names: List[str] = None,
                            include_non_zero_only: bool = False,
                            only_converged: bool = True,
                            verbose: bool = True,
                            unresampled: bool = False) -> pd.DataFrame:
    """
    Compute mean, std and median of metrics across samples for each model.

    Creates a summary table where:
    - Rows: Different metrics (metric_* + time columns)
    - Columns: Model names with Mean, Std, Median subcolumns
    """
    if not csv_filepaths:
        raise ValueError("No CSV files found")

    if model_names is None:
        model_names = [Path(f).stem for f in csv_filepaths]

    if len(csv_filepaths) != len(model_names):
        raise ValueError(f"Mismatch: {len(csv_filepaths)} files vs {len(model_names)} names")

    model_data_metrics = {}
    model_data_full = {}
    all_metrics = set()
    successful_counts = {}

    if verbose:
        print(f"Loading {len(csv_filepaths)} model(s)...")
        print("-" * 80)

    for filepath, model_name in zip(csv_filepaths, model_names):
        df = load_model_results(filepath, verbose=verbose)

        if only_converged and 'converged_train' in df.columns:
            df = df[(df['converged_train'] == True) &
                    (df['converged_second'] == True)].copy()
            if verbose:
                print(f"  {model_name}: filtered to {len(df)} converged_train runs")

        if len(df) > 0:
            metrics_df = extract_metrics(df, include_non_zero_only, unresampled)
            model_data_metrics[model_name] = metrics_df
            model_data_full[model_name] = df
            successful_counts[model_name] = len(metrics_df)
            all_metrics.update(metrics_df.columns)
        else:
            model_data_metrics[model_name] = None
            model_data_full[model_name] = None
            successful_counts[model_name] = 0

    if verbose:
        print("-" * 80)

    all_metrics = sorted(all_metrics)
    summary_rows = []

    # metric_* columns
    for metric in all_metrics:
        row_data = {'Metric': metric}

        for model_name in model_names:
            if model_data_metrics[model_name] is None:
                row_data[f"{model_name}_Mean"] = np.nan
                row_data[f"{model_name}_Std"] = np.nan
                row_data[f"{model_name}_Median"] = np.nan
            else:
                metrics_df = model_data_metrics[model_name]
                if metric in metrics_df.columns:
                    series = metrics_df[metric]
                    row_data[f"{model_name}_Mean"] = series.mean()
                    row_data[f"{model_name}_Std"] = series.std()
                    row_data[f"{model_name}_Median"] = series.median()
                else:
                    row_data[f"{model_name}_Mean"] = np.nan
                    row_data[f"{model_name}_Std"] = np.nan
                    row_data[f"{model_name}_Median"] = np.nan

        summary_rows.append(row_data)

    # timing columns
    for time_col in EXTRA_COLS:
        row_data = {'Metric': time_col}

        for model_name in model_names:
            df_full = model_data_full.get(model_name)
            if df_full is None or time_col not in df_full.columns:
                row_data[f"{model_name}_Mean"] = np.nan
                row_data[f"{model_name}_Std"] = np.nan
                row_data[f"{model_name}_Median"] = np.nan
            else:
                series = df_full[time_col]
                row_data[f"{model_name}_Mean"] = series.mean()
                row_data[f"{model_name}_Std"] = series.std()
                row_data[f"{model_name}_Median"] = series.median()

        summary_rows.append(row_data)

    summary_df = pd.DataFrame(summary_rows)

    summary_df.attrs['model_names'] = model_names
    summary_df.attrs['successful_counts'] = successful_counts
    summary_df.attrs['include_non_zero_only'] = include_non_zero_only
    summary_df.attrs['only_converged'] = only_converged
    summary_df.attrs['unresampled'] = unresampled

    return summary_df


def format_metrics_table(summary_df: pd.DataFrame,
                         model_names: List[str] = None,
                         decimal_places: int = 4,
                         include_non_zero_only: bool = False) -> pd.DataFrame:
    """
    Format metrics summary table as "mean ± std | median" for display.
    """

    if model_names is None:
        model_names = summary_df.attrs.get(
            'model_names',
            list(set([
                col.replace('_Mean', '').replace('_Std', '').replace('_Median', '')
                for col in summary_df.columns if col != 'Metric'
            ]))
        )

    df_display = summary_df.copy()
    if include_non_zero_only:
        df_display = df_display[
            df_display['Metric'].str.contains(
                '_nz|total_count|non_zero_count|non_zero_percentage', na=False
            )
            | df_display['Metric'].isin(EXTRA_COLS)
        ].copy()

    formatted_rows = []

    for _, row in df_display.iterrows():
        formatted_row = {'Metric': row['Metric']}

        for model_name in model_names:
            mean_col = f"{model_name}_Mean"
            std_col = f"{model_name}_Std"
            median_col = f"{model_name}_Median"

            mean_val = row.get(mean_col, np.nan)
            std_val = row.get(std_col, np.nan)
            median_val = row.get(median_col, np.nan)

            if pd.notna(mean_val) and pd.notna(std_val) and pd.notna(median_val):
                formatted_row[model_name] = (
                    f"{mean_val:.{decimal_places}f} ± {std_val:.{decimal_places}f} "
                    f"| {median_val:.{decimal_places}f}"
                )
            else:
                formatted_row[model_name] = "N/A"

        formatted_rows.append(formatted_row)

    return pd.DataFrame(formatted_rows)


def display_metrics(summary_df: pd.DataFrame,
                    model_names: List[str] = None,
                    show_all: bool = False,
                    decimal_places: int = 4) -> pd.DataFrame:
    """
    Display metrics summary as a nice pandas table in Jupyter.

    Parameters
    ----------
    summary_df : pd.DataFrame
        Output from compute_metrics_summary()
    model_names : List[str], optional
        Model names
    show_all : bool
        If False, show only non-zero metrics (recommended)
    decimal_places : int
        Number of decimal places

    Returns
    -------
    pd.DataFrame
        Formatted table for display

    Examples
    --------
    >>> summary = compute_metrics_summary(csv_files, ['LSTM', 'GRU'])
    >>> table = display_metrics(summary, show_all=False)
    >>> display(table)  # In Jupyter
    """

    if model_names is None:
        model_names = summary_df.attrs.get(
            'model_names',
            list(set([
                col.replace('_Mean', '').replace('_Std', '')
                for col in summary_df.columns if col != 'Metric'
            ]))
        )

    # Format table
    formatted = format_metrics_table(
        summary_df,
        model_names,
        decimal_places,
        include_non_zero_only=not show_all,
    )

    # Print summary info
    print(f"Metrics Summary - {'Non-Zero Metrics' if not show_all else 'All Metrics'}")
    print(f"Models: {', '.join(model_names)}")

    if hasattr(summary_df, 'attrs') and 'successful_counts' in summary_df.attrs:
        counts = summary_df.attrs['successful_counts']
        for m in model_names:
            print(f"  {m}: {counts[m]} successful runs")

    print()

    # Configure pandas display
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', None)

    return formatted


def get_metric_comparison(summary_df: pd.DataFrame,
                          metric_name: str,
                          model_names: List[str] = None) -> pd.DataFrame:
    """
    Extract comparison for a specific metric across models.

    Parameters
    ----------
    summary_df : pd.DataFrame
        Output from compute_metrics_summary()
    metric_name : str
        Name of metric to compare
    model_names : List[str], optional
        Model names

    Returns
    -------
    pd.DataFrame
        Comparison table for metric

    Examples
    --------
    >>> mae_nz = get_metric_comparison(summary, 'metric_mae_nz')
    >>> display(mae_nz)
    """

    if model_names is None:
        model_names = summary_df.attrs.get(
            'model_names',
            list(set([
                col.replace('_Mean', '').replace('_Std', '')
                for col in summary_df.columns if col != 'Metric'
            ]))
        )

    metric_row = summary_df[summary_df['Metric'] == metric_name]

    if len(metric_row) == 0:
        print(f"✗ Metric '{metric_name}' not found")
        print("\nAvailable metrics:")
        for i, m in enumerate(sorted(summary_df['Metric'].unique()), 1):
            print(f"  {i:2}. {m}")
        return None

    comparison = [{'Metric': metric_name}]

    for model_name in model_names:
        mean_val = metric_row[f"{model_name}_Mean"].values[0]
        std_val = metric_row[f"{model_name}_Std"].values[0]

        if pd.notna(mean_val):
            comparison[0][model_name] = f"{mean_val:.6f} ± {std_val:.6f}"
        else:
            comparison[0][model_name] = "N/A"

    return pd.DataFrame(comparison)


def list_available_metrics(summary_df: pd.DataFrame) -> None:
    """
    Display list of available metrics.

    Parameters
    ----------
    summary_df : pd.DataFrame
        Output from compute_metrics_summary()
    """
    metrics = sorted(summary_df['Metric'].unique())

    print(f"Available metrics ({len(metrics)} total):\n")

    for i, m in enumerate(metrics, 1):
        print(f"  {i:2}. {m}")


# ============================================================================
# EXAMPLE USAGE IN JUPYTER NOTEBOOK
# ============================================================================
"""
Example Usage:
==============

# 1. Load results from multiple models
csv_files = ['lstm_results.csv', 'gru_results.csv', 'transformer_results.csv']
model_names = ['LSTM', 'GRU', 'Transformer']

# 2. Compute summary statistics
summary = compute_metrics_summary(csv_files, model_names)

# 3. Display as formatted pandas table (non-zero metrics + time columns)
table = display_metrics(summary, show_all=False)
display(table)  # Use IPython display in Jupyter

# 4. Display all metrics
table_all = display_metrics(summary, show_all=True)
display(table_all)

# 5. Compare specific metric or time column
mae_comparison = get_metric_comparison(summary, 'metric_mae_nz')
display(mae_comparison)

train_time_cmp = get_metric_comparison(summary, 'train_time_seconds')
display(train_time_cmp)

# 6. List available metricsb
list_available_metrics(summary)
"""


"\nExample Usage:\n==============\n\n# 1. Load results from multiple models\ncsv_files = ['lstm_results.csv', 'gru_results.csv', 'transformer_results.csv']\nmodel_names = ['LSTM', 'GRU', 'Transformer']\n\n# 2. Compute summary statistics\nsummary = compute_metrics_summary(csv_files, model_names)\n\n# 3. Display as formatted pandas table (non-zero metrics + time columns)\ntable = display_metrics(summary, show_all=False)\ndisplay(table)  # Use IPython display in Jupyter\n\n# 4. Display all metrics\ntable_all = display_metrics(summary, show_all=True)\ndisplay(table_all)\n\n# 5. Compare specific metric or time column\nmae_comparison = get_metric_comparison(summary, 'metric_mae_nz')\ndisplay(mae_comparison)\n\ntrain_time_cmp = get_metric_comparison(summary, 'train_time_seconds')\ndisplay(train_time_cmp)\n\n# 6. List available metricsb\nlist_available_metrics(summary)\n"

In [6]:
from pathlib import Path
from typing import List
import numpy as np
import pandas as pd

# Extra columns for timing information
EXTRA_COLS = [
    "train_train_time_seconds",
    "second_train_time_seconds",
    "prediction_time_seconds",
    "number_of_anomalies",
    "number_of_anomalies_robust",
]

def load_model_results(csv_filepath: str, verbose: bool = True) -> pd.DataFrame:
    """
    Load and filter model results from CSV file.

    Returns only successful runs.
    """
    try:
        df = pd.read_csv(csv_filepath)
        successful = df[df["status"] == "success"].copy()

        if verbose and len(successful) > 0:
            print(f"✓ Loaded {len(successful)} successful runs from {csv_filepath}")

        return successful
    except Exception as e:
        print(f"✗ Error loading {csv_filepath}: {e}")
        return pd.DataFrame()

def extract_metrics(
    df: pd.DataFrame,
    include_non_zero_only: bool = False,
    unresampled: bool = False,
) -> pd.DataFrame:
    """
    Extract metric columns from results dataframe.

    Parameters
    ----------
    df : pd.DataFrame
        Results dataframe
    include_non_zero_only : bool
        If True, extract only _nz metrics and summary count metrics
    unresampled : bool
        If True, use unresampled_metric_* columns, otherwise metric_* columns
    """
    if unresampled:
        metric_cols = [col for col in df.columns if col.startswith("unresampled_metric_")]
        total_count_col = "unresampled_metric_total_count"
        non_zero_count_col = "unresampled_metric_non_zero_count"
        zero_count_col = "unresampled_metric_zero_count"
        non_zero_percentage_col = "unresampled_metric_non_zero_percentage"
    else:
        metric_cols = [col for col in df.columns if col.startswith("metric_")]
        total_count_col = "metric_total_count"
        non_zero_count_col = "metric_non_zero_count"
        zero_count_col = "metric_zero_count"
        non_zero_percentage_col = "metric_non_zero_percentage"

    if include_non_zero_only:
        metric_cols = [
            col
            for col in metric_cols
            if col.endswith("_nz")
            or col in [
                total_count_col,
                non_zero_count_col,
                zero_count_col,
                non_zero_percentage_col,
            ]
        ]

    return df[metric_cols]

def compute_metrics_summary(
    csv_filepaths: List[str],
    model_names: List[str] = None,
    include_non_zero_only: bool = False,
    only_converged: bool = True,
    verbose: bool = True,
    unresampled: bool = False,
) -> pd.DataFrame:
    """
    Compute mean, std and median of metrics across samples for each model.

    Important:
    Only considers filenames that are successful and converged in all models.
    Metric type is controlled by the unresampled flag.
    """
    if not csv_filepaths:
        raise ValueError("No CSV files found")

    if model_names is None:
        model_names = [Path(f).stem for f in csv_filepaths]

    if len(csv_filepaths) != len(model_names):
        raise ValueError(
            f"Mismatch: {len(csv_filepaths)} files vs {len(model_names)} names"
        )

    model_data_metrics = {}
    model_data_full = {}
    all_metrics = set()
    successful_counts = {}
    per_model_filenames = {}

    if verbose:
        print(f"Loading {len(csv_filepaths)} model(s)...")
        print("-" * 80)

    for filepath, model_name in zip(csv_filepaths, model_names):
        df = load_model_results(filepath, verbose=verbose)

        if only_converged:
            if "converged_train" not in df.columns or "converged_second" not in df.columns:
                raise ValueError(
                    f"Columns 'converged_train' and/or 'converged_second' not found in {filepath}"
                )
            df = df[
                (df["converged_train"] == True) &
                (df["converged_second"] == True)
            ].copy()

            if verbose:
                print(f"  {model_name}: filtered to {len(df)} converged runs")

        if "filename" not in df.columns:
            raise ValueError(f"'filename' column not found in {filepath}")

        per_model_filenames[model_name] = set(df["filename"].dropna().unique())
        model_data_full[model_name] = df

    if not per_model_filenames:
        raise ValueError("No models loaded")

    common_filenames = set.intersection(*per_model_filenames.values())

    if verbose:
        print("-" * 80)
        print(f"Common filenames across all models (successful + converged): {len(common_filenames)}")

    for model_name in model_names:
        df = model_data_full[model_name]
        df = df[df["filename"].isin(common_filenames)].copy()

        if len(df) > 0:
            metrics_df = extract_metrics(
                df,
                include_non_zero_only=include_non_zero_only,
                unresampled=unresampled,
            )
            model_data_metrics[model_name] = metrics_df
            model_data_full[model_name] = df
            successful_counts[model_name] = len(metrics_df)
            all_metrics.update(metrics_df.columns)

            if verbose:
                print(f"  {model_name}: {len(metrics_df)} rows after filename intersection")
        else:
            model_data_metrics[model_name] = None
            model_data_full[model_name] = None
            successful_counts[model_name] = 0

            if verbose:
                print(f"  {model_name}: 0 rows after filename intersection")

    if verbose:
        print("-" * 80)

    all_metrics = sorted(all_metrics)
    summary_rows = []

    for metric in all_metrics:
        row_data = {"Metric": metric}

        for model_name in model_names:
            if model_data_metrics[model_name] is None:
                row_data[f"{model_name}_Mean"] = np.nan
                row_data[f"{model_name}_Std"] = np.nan
                row_data[f"{model_name}_Median"] = np.nan
            else:
                metrics_df = model_data_metrics[model_name]
                if metric in metrics_df.columns:
                    series = metrics_df[metric]
                    row_data[f"{model_name}_Mean"] = series.mean()
                    row_data[f"{model_name}_Std"] = series.std()
                    row_data[f"{model_name}_Median"] = series.median()
                else:
                    row_data[f"{model_name}_Mean"] = np.nan
                    row_data[f"{model_name}_Std"] = np.nan
                    row_data[f"{model_name}_Median"] = np.nan

        summary_rows.append(row_data)

    for time_col in EXTRA_COLS:
        row_data = {"Metric": time_col}

        for model_name in model_names:
            df_full = model_data_full.get(model_name)
            if df_full is None or time_col not in df_full.columns:
                row_data[f"{model_name}_Mean"] = np.nan
                row_data[f"{model_name}_Std"] = np.nan
                row_data[f"{model_name}_Median"] = np.nan
            else:
                series = df_full[time_col]
                row_data[f"{model_name}_Mean"] = series.mean()
                row_data[f"{model_name}_Std"] = series.std()
                row_data[f"{model_name}_Median"] = series.median()

        summary_rows.append(row_data)

    summary_df = pd.DataFrame(summary_rows)

    summary_df.attrs["model_names"] = model_names
    summary_df.attrs["successful_counts"] = successful_counts
    summary_df.attrs["include_non_zero_only"] = include_non_zero_only
    summary_df.attrs["only_converged"] = only_converged
    summary_df.attrs["unresampled"] = unresampled
    summary_df.attrs["common_filenames_count"] = len(common_filenames)

    return summary_df

def format_metrics_table(
    summary_df: pd.DataFrame,
    model_names: List[str] = None,
    decimal_places: int = 4,
    include_non_zero_only: bool = False,
) -> pd.DataFrame:
    """
    Format metrics summary table as 'mean ± std | median' for display.
    """
    if model_names is None:
        model_names = summary_df.attrs.get(
            "model_names",
            list(
                set(
                    [
                        col.replace("_Mean", "")
                        .replace("_Std", "")
                        .replace("_Median", "")
                        for col in summary_df.columns
                        if col != "Metric"
                    ]
                )
            ),
        )

    df_display = summary_df.copy()

    if include_non_zero_only:
        df_display = df_display[
            df_display["Metric"].str.contains(
                "_nz|total_count|non_zero_count|non_zero_percentage", na=False
            )
            | df_display["Metric"].isin(EXTRA_COLS)
        ].copy()

    formatted_rows = []

    for _, row in df_display.iterrows():
        formatted_row = {"Metric": row["Metric"]}

        for model_name in model_names:
            mean_col = f"{model_name}_Mean"
            std_col = f"{model_name}_Std"
            median_col = f"{model_name}_Median"

            mean_val = row.get(mean_col, np.nan)
            std_val = row.get(std_col, np.nan)
            median_val = row.get(median_col, np.nan)

            if pd.notna(mean_val) and pd.notna(std_val) and pd.notna(median_val):
                formatted_row[model_name] = (
                    f"{mean_val:.{decimal_places}f} ± "
                    f"{std_val:.{decimal_places}f} | "
                    f"{median_val:.{decimal_places}f}"
                )
            else:
                formatted_row[model_name] = "N/A"

        formatted_rows.append(formatted_row)

    return pd.DataFrame(formatted_rows)

def display_metrics(
    summary_df: pd.DataFrame,
    model_names: List[str] = None,
    show_all: bool = False,
    decimal_places: int = 4,
) -> pd.DataFrame:
    """
    Display metrics summary as a nice pandas table in Jupyter.
    """
    if model_names is None:
        model_names = summary_df.attrs.get(
            "model_names",
            list(
                set(
                    [
                        col.replace("_Mean", "").replace("_Std", "")
                        for col in summary_df.columns
                        if col != "Metric"
                    ]
                )
            ),
        )

    formatted = format_metrics_table(
        summary_df,
        model_names,
        decimal_places,
        include_non_zero_only=not show_all,
    )

    metric_mode = "unresampled" if summary_df.attrs.get("unresampled", False) else "regular"

    print(f"Metrics Summary - {'Non-Zero Metrics' if not show_all else 'All Metrics'}")
    print(f"Metric type: {metric_mode}")
    print(f"Models: {', '.join(model_names)}")

    if "common_filenames_count" in summary_df.attrs:
        print(f"Common filenames in all models: {summary_df.attrs['common_filenames_count']}")

    if "successful_counts" in summary_df.attrs:
        counts = summary_df.attrs["successful_counts"]
        for m in model_names:
            print(f"  {m}: {counts[m]} rows after common filename filtering")

    print()

    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", None)
    pd.set_option("display.max_colwidth", None)

    return formatted

def get_metric_comparison(
    summary_df: pd.DataFrame,
    metric_name: str,
    model_names: List[str] = None,
) -> pd.DataFrame:
    """
    Extract comparison for a specific metric across models.
    """
    if model_names is None:
        model_names = summary_df.attrs.get(
            "model_names",
            list(
                set(
                    [
                        col.replace("_Mean", "").replace("_Std", "")
                        for col in summary_df.columns
                        if col != "Metric"
                    ]
                )
            ),
        )

    metric_row = summary_df[summary_df["Metric"] == metric_name]

    if len(metric_row) == 0:
        print(f"✗ Metric '{metric_name}' not found")
        print("\nAvailable metrics:")
        for i, m in enumerate(sorted(summary_df["Metric"].unique()), 1):
            print(f"  {i:2}. {m}")
        return None

    comparison = [{"Metric": metric_name}]

    for model_name in model_names:
        mean_val = metric_row[f"{model_name}_Mean"].values[0]
        std_val = metric_row[f"{model_name}_Std"].values[0]

        if pd.notna(mean_val):
            comparison[0][model_name] = f"{mean_val:.6f} ± {std_val:.6f}"
        else:
            comparison[0][model_name] = "N/A"

    return pd.DataFrame(comparison)

def list_available_metrics(summary_df: pd.DataFrame) -> None:
    """
    Display list of available metrics.
    """
    metrics = sorted(summary_df["Metric"].unique())

    print(f"Available metrics ({len(metrics)} total):\n")

    for i, m in enumerate(metrics, 1):
        print(f"  {i:2}. {m}")

## Metrics table comparison

In [7]:
csv_files = ['./6_weeks_results_seasonal_uc_1000_seed_42_daily_clipped_tree_timeout_120_reworked_all.csv',
             './6_weeks_results_seasonal_uc_1000_seed_42_daily_clipped_tree_timeout_120_reworked_daily_all.csv',
             './6_weeks_results_seasonal_uc_1000_seed_42_daily_clipped_tree_timeout_120_reworked_daily_weekly_fourier_all.csv',
             '6_weeks_results_seasonal_uc_1000_seed_42_daily_clipped_tree_timeout_120_reworked_daily_steps_weekly_fourier_all.csv']

model_names = ['Local Level Only', 'Daily Seasonal', "Daily + Weekly Fourier", "Daily Steps + Weekly Fourier"]

# 2. Compute summary statistics
summary = compute_metrics_summary(csv_files, model_names=model_names, unresampled=True, only_converged=True)
table = display_metrics(summary, show_all=True)
display(table)

Loading 4 model(s)...
--------------------------------------------------------------------------------
✓ Loaded 19629 successful runs from ./6_weeks_results_seasonal_uc_1000_seed_42_daily_clipped_tree_timeout_120_reworked_all.csv
  Local Level Only: filtered to 15227 converged runs
✓ Loaded 19629 successful runs from ./6_weeks_results_seasonal_uc_1000_seed_42_daily_clipped_tree_timeout_120_reworked_daily_all.csv
  Daily Seasonal: filtered to 13607 converged runs
✓ Loaded 19629 successful runs from ./6_weeks_results_seasonal_uc_1000_seed_42_daily_clipped_tree_timeout_120_reworked_daily_weekly_fourier_all.csv
  Daily + Weekly Fourier: filtered to 11531 converged runs
✓ Loaded 19629 successful runs from 6_weeks_results_seasonal_uc_1000_seed_42_daily_clipped_tree_timeout_120_reworked_daily_steps_weekly_fourier_all.csv
  Daily Steps + Weekly Fourier: filtered to 12601 converged runs
--------------------------------------------------------------------------------
Common filenames across all 

,Metric,Local Level Only,Daily Seasonal,Daily + Weekly Fourier,Daily Steps + Weekly Fourier
0,unresampled_metric_direction_accuracy,0.5356 ± 0.1404 | 0.5098,0.5910 ± 0.0923 | 0.5875,0.5911 ± 0.0920 | 0.5933,0.5915 ± 0.0901 | 0.5890
1,unresampled_metric_direction_accuracy_nz,0.4797 ± 0.1105 | 0.4783,0.5679 ± 0.1000 | 0.5669,0.5719 ± 0.1081 | 0.5750,0.5690 ± 0.0999 | 0.5685
2,unresampled_metric_mae,0.1978 ± 1.4025 | 0.0631,43847.8133 ± 3811283.7061 | 0.0608,5209.5132 ± 452840.8405 | 0.0541,67810.3922 ± 5894384.7281 | 0.0612
3,unresampled_metric_mae_nz,0.2417 ± 1.6479 | 0.0705,65893.6977 ± 5679083.1681 | 0.0678,6373.7336 ± 549306.0216 | 0.0596,99086.3630 ± 8539820.8158 | 0.0682
4,unresampled_metric_mape,68245166108147.1250 ± 1200948337140838.5000 | 3385169639735.2881,77175764965672615936.0000 ± 6707014182485166653440.0000 | 2809650758995.2251,11825206099216965632.0000 ± 1027919983773053353984.0000 | 2428203836868.7705,124496518647911940096.0000 ± 10820695661034386489344.0000 | 2929407828835.8828
5,unresampled_metric_mape_nz,1.4030 ± 7.0701 | 0.8021,27975378.8528 ± 2411081562.6751 | 0.7953,2110535.9566 ± 181898199.0367 | 0.6865,31676134.9900 ± 2730034356.9274 | 0.8012
6,unresampled_metric_max_residual,0.0000 ± 0.0000 | 0.0000,0.0000 ± 0.0000 | 0.0000,0.0000 ± 0.0000 | 0.0000,0.0000 ± 0.0000 | 0.0000
7,unresampled_metric_max_residual_nz,0.0000 ± 0.0000 | 0.0000,0.0000 ± 0.0000 | 0.0000,0.0000 ± 0.0000 | 0.0000,0.0000 ± 0.0000 | 0.0000
8,unresampled_metric_mean_residual,0.0000 ± 0.0000 | 0.0000,0.0000 ± 0.0000 | 0.0000,0.0000 ± 0.0000 | 0.0000,0.0000 ± 0.0000 | 0.0000
9,unresampled_metric_mean_residual_nz,0.0000 ± 0.0000 | 0.0000,0.0000 ± 0.0000 | 0.0000,0.0000 ± 0.0000 | 0.0000,0.0000 ± 0.0000 | 0.0000


In [ ]:
dfs = [pd.read_csv(f) for f in csv_files]
dfs
#plot_histograms_with_stats(
#    dfs=dfs,
#    metric="unresampled_metric_mae",
#    titles=model_names
#)

In [2]:
print(dfs)

NameError: name 'dfs' is not defined